# Neo4j 연결 확인
- Neo4j Python Driver로 DB에 연결하고 `verify_connectivity()`로 접속 정보를 확인

## Neo4j란?

- 데이터를 **노드(Node)** 와 **관계(Relationship)** 로 저장하는 그래프 데이터베이스
- 테이블과 외래 키를 중심으로 데이터를 표현하는 관계형 데이터베이스와 달리, 데이터 사이의 연결을 관계 자체로 저장하므로 여러 단계를 따라가는 탐색에 적합함
- neo4j Desktop 다운로드 링크 : https://neo4j.com/download/
- neo4j Desktop 설치 가이드 : https://giant-tadpole-370.notion.site/Neo4j-Desktop-3b21178026258312b4fd014c722043bc?source=copy_link

### Property Graph의 구성 요소

```text
(고객 A:Customer)-[:PLACED]->(주문 O-1001:Order)
```

| 구성 요소 | 설명 | 쇼핑몰 예시 |
|---|---|---|
| 노드 | 현실 세계의 개체 | 고객, 주문, 상품, 물류센터 |
| 라벨 | 노드의 종류 | `Customer`, `Order`, `Product` |
| 속성 | 노드나 관계의 세부 값 | `id`, `name`, `type` |
| 관계 | 노드 사이의 연결과 방향 | `PLACED`, `CONTAINS`, `SHIPPED_BY` |

관계도 속성을 가질 수 있습니다. 예를 들어 배송 관계에 `배송일`, `상태` 같은 값을 저장할 수 있습니다.

### Cypher

- Neo4j는 그래프를 조회하고 변경할 때 **Cypher** 라는 선언형 쿼리 언어를 사용함
- 괄호는 노드, 대괄호와 화살표는 관계를 나타냄

```text
MATCH (customer:Customer {id: "고객 A"})
      -[:PLACED]->(order:Order)
RETURN order.id
```

이 쿼리는 `고객 A`가 생성한 주문을 관계 방향을 따라 조회합니다.

### Graph RAG에서 Neo4j를 사용하는 이유

- 고객 → 주문 → 상품처럼 여러 관계를 연결하는 **다중 홉 탐색**에 유리합니다.
- 문서 청크뿐 아니라 엔티티와 관계를 함께 검색할 수 있습니다.
- 어떤 경로로 답을 찾았는지 확인할 수 있어 답변의 근거를 설명하기 쉽습니다.
- LLM이 자연어 질문을 Cypher로 변환하면 복잡한 연결 관계도 질의할 수 있습니다.

### 이번 실습에서 사용하는 접속 방식

- **Neo4j Browser**: 브라우저에서 Cypher를 실행하고 그래프 시각화
- **Bolt 프로토콜**: Python 애플리케이션이 Neo4j에 접속할 때 사용
- **Neo4j Python Driver**: 연결 확인과 Cypher 실행 담당
- **LangChain `Neo4jGraph`**: 이후 노트북에서 스키마 조회와 Graph RAG 체인 구성

## 환경

### 1) Neo4j

Neo4j Desktop의 DB를 실행 합니다.

- Neo4j Browser: `http://localhost:7474`
- Bolt: `bolt://localhost:7687`
- 사용자: `neo4j`
- 비밀번호: `graphragpassword`

### 2) `.env`

```env
OPENAI_API_KEY=your_openai_api_key
OPENAI_MODEL=gpt-5.5

NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=graphragpassword
NEO4J_DATABASE=neo4j
```



## 1. 환경변수 읽기

비밀번호 값은 출력하지 않고 설정 여부만 확인합니다.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

URI = os.environ["NEO4J_URI"]
USERNAME = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

print("NEO4J_URI:", URI or "(미설정)")
print("NEO4J_USERNAME:", USERNAME or "(미설정)")
print("NEO4J_PASSWORD 설정:", bool(PASSWORD))
print("NEO4J_DATABASE:", DATABASE)

NEO4J_URI: bolt://localhost:7687
NEO4J_USERNAME: neo4j
NEO4J_PASSWORD 설정: True
NEO4J_DATABASE: neo4j


## 2. 연결 함수

`driver`는 Neo4j 서버와 통신하는 연결 관리자입니다. 사용 후 반드시 닫습니다.

In [4]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

In [5]:
try:
    # 인증, 네트워크, 서버 응답까지 포함한 실제 연결 검사 수행
    driver.verify_connectivity()
    # 연결된 서버 정보 읽어와 인스턴스인지 확인
    server = driver.get_server_info()
    print("server:", server.agent)
finally:
    driver.close()

server: Neo4j/2026.06.0
